# Economy-wide impact assessment

This tutorial accompanies the [CVXLab economy-wide impact assessment](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html).
Both paths use Leontief input-output analysis and retain twelve electricity-generation technologies.

- The MARIO tutorial prepares the Italian reference, updates its electricity mix to 2022,
  replaces direct electricity-emission intensities using the public EXIOBASE 3.11.2 IOT,
  and calculates the electricity scenarios.
- The CVXLab tutorial formulates the model, solves the scenarios and analyses the results.

Complete the common preparation, then continue in MARIO or export to CVXLab.
Familiarity with input-output analysis is assumed. Use Python 3.12 and MARIO 1.3.5,
installed as shown under Tutorial materials. The optional CVXLab path uses the revision
identified at the branch point.

## Problem statement

How do changes to Italy's electricity supply propagate to domestic value added,
imports and greenhouse-gas emissions?

Compare Reference with **High_RES**, which applies the High RES supply shares below,
and **High_RES_domestic**, which also replaces intermediate and final electricity imports
with domestic electricity. Fuel and other imports follow domestic producers' input needs.
Calculate percentage changes from Reference and compare the two High RES scenarios.

| Electricity generation by | Reference (2022) | High RES |
| --- | ---: | ---: |
| Coal | 8.07%| 0.0% |
| Gas | 50.49%| 0.0% |
| Nuclear | 0.00%| 15.0% |
| Hydro | 10.14%| 9.0% |
| Wind | 7.31%| 38.0% |
| Petroleum | 4.61%| 0.0% |
| Biomass and waste | 6.29%| 5.0% |
| Solar photovoltaic | 10.04%| 31.0% |
| Tide, wave, ocean | 0.00%| 0.5% |
| Geothermal | 2.08%| 1.5% |
| Others | 0.97%| 0.0% |

Shares refer to monetary production within electricity-generation activities.
Reference values reflect the 2022 mix update applied to the Italian SUT below. Others combines solar
thermal and the remaining electricity activity. Secondary electricity production by
other industries retains its reference market share. High RES is the shared teaching scenario.

## Tutorial materials

This notebook contains the full workflow. Download it, `aggregation.xlsx`, `scenario_summary.csv` and the optional `results.zip` and `cvxlab_model.zip`
from the [MARIO tutorial materials](https://tutorials.marioproject.com/economy-wide-impact-assessment/index.html#tutorial-materials)
into one working folder. The [CVXLab materials](https://cvxlab.readthedocs.io/en/dev/_downloads/f16216b31c3f637ad45967d9f5ee116d/ioa_mario.zip) contain its conceptual
workbook and receiving model configuration for the optional export path.

Create and activate a dedicated environment before installing the MARIO 1.3.5
release. With an existing environment, start at `conda activate`.
Launch Jupyter in the tutorial folder.

```sh
conda create -n mario_cvxlab_tutorial python=3.12 pip
conda activate mario_cvxlab_tutorial
python -m pip install "mariopy==1.3.5" jupyterlab
python -m jupyter lab
```

The next section obtains the source archives under `source_data`. The completed
MARIO path writes the scenario matrices, `sets.xlsx` and a summary under `results`.
The published `results.zip` contains nine flow matrices, matching sets and the summary
for all three scenarios. Activity output `Xa` and emission-intensity tables are not
included in the published files. The notebook generates `Xa` locally for CVXLab.
For the CVXLab path, extract `cvxlab_model.zip` beside this notebook. It supplies
the receiving settings, classified sets, reference flows and scenario shocks.
Its `total_activity_0` values are left blank; running `to_cvxlab` fills them locally.

## Database download

Download the monetary EXIOBASE 3.8.2 SUT for 2022 and its matching industry-by-industry
IOT satellite accounts. MARIO checks the published size and checksum and reuses valid
existing archives. Keep both archives zipped for parsing. The downloads total approximately 1 GB.

The files are also available from the [official EXIOBASE record](https://zenodo.org/records/5589597).
Place manually downloaded `MRSUT_2022.zip` and `IOT_2022_ixi.zip` under `source_data`.
EXIOBASE 3.8.2 was published in 2021 and its 2022 tables are estimated extensions of
the historical series. Retain the source's CC-BY-SA-4.0 licence and
[EXIOBASE attribution](https://doi.org/10.1111/jiec.12715) when reusing derived results.


The emission-correction step also downloads the public EXIOBASE 3.11.2 ixi IOT
for 2022 into `source_data/3.11.2`. Keeping releases in separate directories avoids
a collision between the two files named `IOT_2022_ixi.zip`. Its source licence
and attribution are linked in that step. No private 3.11.2 SUT is needed.

In [ ]:
from pathlib import Path
import inspect

import mario
import numpy as np
import pandas as pd

assert "trade_resolution" in inspect.signature(mario.Database.to_single_region).parameters, (
    "Install MARIO 1.3.5 as shown above and restart the kernel."
)

source_dir = Path("source_data")
output_dir = Path("results")
sut_path = source_dir / "MRSUT_2022.zip"
iot_path = source_dir / "IOT_2022_ixi.zip"

In [ ]:
mario.download_exiobase3(
    path=source_dir,
    years=[2022],
    table="SUT",
    version="3.8.2",
    extract=False,
)
mario.download_exiobase3(
    path=source_dir,
    years=[2022],
    table="IOT",
    system="ixi",
    version="3.8.2",
    extract=False,
)
assert sut_path.is_file() and iot_path.is_file()

## Database parsing

The SUT archive supplies economic flows. Satellite accounts come from the industry-by-industry
IOT of the same release and year. MARIO attaches them to the corresponding activities.

In [ ]:
db = mario.parse_exiobase(
    path=str(sut_path),
    table="SUT",
    unit="Monetary",
    add_extensions=str(iot_path),
)

## Transformation to single region

`import_representation="separate"` creates native import matrices alongside the factors
of production. `imports_by="commodity"` retains the imported-product classification.
`trade_resolution="aggregate"` combines foreign origins and export destinations.

`IMP_a` records imports used by activities, `IMP_c` imports entering domestic commodity
supply and `IMP_Y` imports delivered directly to final demand. `Va` retains the
domestic factors of production. Imported inputs are excluded from `U` and `Yc`.

In [ ]:
db.to_single_region(
    "IT",
    imports_by="commodity",
    import_representation="separate",
    trade_resolution="aggregate",
)

## Aggregation

Map the twelve EXIOBASE electricity commodities to `Electricity`, then apply
`aggregation.xlsx` to the activities and remaining products. This aggregation mapping
keeps all twelve generation technologies separate so the CVXLab exercise can vary
their shares. It produces 18 products and 29 activities. Transmission and distribution
share one activity. The commodity mapping also applies to import accounts.
MARIO reads the `Aggregation` column. `Aggregation II` supplies the five reporting
groups used by the receiving CVXLab model.
`zero_output_epsilon=None` sums the observed flows directly.

In [ ]:
commodities = db.get_index("Commodity")
electricity = db.search("Commodity", "electricity ")
assert len(electricity) == 12

commodity_mapping = pd.DataFrame({"Aggregation": commodities}, index=commodities)
commodity_mapping.loc[electricity, "Aggregation"] = "Electricity"
db.aggregate(
    io={"Commodity": commodity_mapping},
    levels=["Commodity"],
    calc_all=False,
    zero_output_epsilon=None,
)

aggregation_path = Path("aggregation.xlsx")
for level in ["Activity", "Commodity"]:
    mapping = pd.read_excel(aggregation_path, sheet_name=level, index_col=0)
    assert mapping.index.is_unique
    assert set(mapping.index) == set(db.get_index(level))
    assert mapping["Aggregation"].notna().all()
    assert mapping["Aggregation"].astype(str).str.strip().ne("").all()
    if level == "Activity":
        generation = mapping.index.str.startswith("Production of electricity")
        assert generation.sum() == 12
        assert mapping.loc[generation, "Aggregation"].tolist() == mapping.index[generation].tolist()

db.aggregate(
    io=str(aggregation_path),
    levels=["Activity", "Commodity"],
    calc_all=False,
    zero_output_epsilon=None,
)

## GHG characterization

Combine the EXIOBASE greenhouse-gas accounts with the built-in IPCC AR6 GWP100 profile.
The retained row measures kg CO2-equivalent. Direct final-demand emissions remain in `EY`.

In [ ]:
ghg_label = "GHG AR6 GWP-100"
db.calc_ghg(profile="exiobase_monetary", label=ghg_label)
db.keep_extensions([ghg_label])

### Check the dimensions and import coefficients

These checks verify the reference classification with the twelve native generation technologies. MARIO computes
`imp_a` by dividing each column of `IMP_a` by the corresponding activity output `Xa`.

In [ ]:
expected_sizes = {
    "Region": 1,
    "Commodity": 18,
    "Activity": 29,
    "Consumption category": 9,
    "Factor of production": 12,
    "Satellite account": 1,
}
sizes = {name: len(db.get_index(name)) for name in expected_sizes}
assert sizes == expected_sizes, sizes
assert "imports" not in db.get_index("Factor of production")

imports = db.query("IMP_a")
coefficients = db.query("imp_a")
activity_output = db.query("Xa").iloc[:, 0]
assert imports.columns.equals(activity_output.index)
assert imports.index.equals(coefficients.index)
np.testing.assert_allclose(
    coefficients.to_numpy() * activity_output.to_numpy(),
    imports.to_numpy(),
    rtol=1e-10,
    atol=1e-6,
)
pd.Series(sizes, name="Members").to_frame()

## Inspect the original electricity data

Inspect the original generation shares and direct combustion-GHG intensities before
calibrating the reference. In this source, photovoltaic production is very small and
its direct intensity is unusually high; solar thermal has negligible output and an
extreme direct intensity. These observations motivate the two corrections below.
The displayed intensities in this cell belong only to the original EXIOBASE 3.8.2 data.

In [ ]:
power_activities = db.search("Activity", "Production of electricity")
power_columns = [("IT", "Activity", name) for name in power_activities]
electricity_column = ("IT", "Commodity", "Electricity")

original_mix = db.query("s").loc[power_columns, electricity_column].droplevel([0, 1])
original_mix = original_mix / original_mix.sum() * 100
original_intensities = db.query("ea").loc[ghg_label, power_columns].droplevel([0, 1])
display(original_mix.to_frame("Original generation share (%)"))
display(original_intensities.to_frame("Original direct intensity (kg CO2eq/EUR)"))

## Update the reference electricity mix to 2022

Use MARIO's electricity preset to update the baseline supply shares from EMBER's
2022 generation data. This changes the reference before the two teaching scenarios.
The preset maps generation technologies to EXIOBASE activities, retains secondary
electricity production and rebuilds the economic flows. Its physical generation
shares are applied as monetary supply-share targets in this monetary exercise;
technology-specific electricity prices are not modelled.

See the [electricity preset guide](https://www.marioproject.com/en/latest/user_guide/advanced/electricity_mix.html).

In [ ]:
db.update_supply_mix("electricity", year=2022)

reference_electricity_shares = db.query("s").loc[power_columns, electricity_column].droplevel([0, 1])
reference_electricity_shares = reference_electricity_shares / reference_electricity_shares.sum() * 100
mix_comparison = pd.concat([
    original_mix.rename("Original EXIOBASE 3.8.2 (%)"),
    reference_electricity_shares.rename("Updated reference, 2022 (%)"),
], axis=1)
display(mix_comparison)

## Update direct electricity-emission intensities

Keep the EXIOBASE 3.8.2 economic SUT and take the direct monetary emission coefficients
for all twelve Italian generation technologies from the public EXIOBASE 3.11.2
industry-by-industry IOT for the same year, 2022. The IOT uses M.EUR while the SUT uses EUR. Divide donor `e` by 1,000,000
before assigning it to `ea`, so both coefficients use kg CO2-equivalent/EUR.
Apply the same `exiobase_monetary` AR6 GWP100 profile to both: the indicator covers
the profile's combustion CO2, CH4 and N2O accounts, rather than all source emissions.
These are direct coefficients; upstream effects continue to follow the SUT's input recipes.

The following cells obtain the coefficients at execution time without displaying
or embedding their numerical values in this notebook. Source archives and extracted
intensities remain local. EXIOBASE 3.11.2 has its own non-commercial licence;
check its conditions for your use of the source data and locally generated exports.
[EXIOBASE 3.11.2](https://zenodo.org/records/23039074),
[source licence](https://zenodo.org/records/23039074/files/LICENSE.txt?download=1),
[EXIOBASE attribution](https://doi.org/10.1111/jiec.12715).

In [ ]:
donor_dir = source_dir / "3.11.2"
mario.download_exiobase3(
    path=donor_dir,
    years=[2022],
    table="IOT",
    system="ixi",
    version="3.11.2",
    extract=False,
)
donor = mario.parse_exiobase(
    path=str(donor_dir / "IOT_2022_ixi.zip"),
    table="IOT",
    unit="Monetary",
)
donor.calc_ghg(profile="exiobase_monetary", label=ghg_label)
donor_columns = [("IT", "Sector", activity) for activity in power_activities]
# Convert kg CO2eq/M.EUR in the IOT to kg CO2eq/EUR in the SUT.
donor_intensities = donor.query("e").loc[ghg_label, donor_columns].droplevel([0, 1]) / 1_000_000
del donor

### Apply the intensities in Python

Copy `ea` and replace only the twelve generation-activity coefficients with the
values extracted above. Reset the baseline to coefficients before storing the
updated matrix: this clears previously materialized flows so `Ea` will be rebuilt
from the new coefficients and activity output when queried or exported.
Economic coefficients, separate imports and direct final-demand emissions retain
their post-mix values. No Excel shock workbook is needed and the updated
intensity values are not displayed.

See the [scenario-update API](https://www.marioproject.com/en/latest/api_document/mario.Database.update_scenarios.html)
and [coefficient reset API](https://www.marioproject.com/en/latest/api_document/mario.Database.reset_to_coefficients.html).

In [ ]:
ea = db.query("ea").copy()
ea.loc[ghg_label, power_columns] = donor_intensities.reindex(power_activities).to_numpy()
db.reset_to_coefficients("baseline")
db.update_scenarios("baseline", ea=ea)

## Continue in MARIO or CVXLab

The prepared reference is the common starting point. Continue with the MARIO
scenarios below, or export it to CVXLab and follow its model-building tutorial.

Set `continue_in_cvxlab = True` to refresh the CVXLab reference with `to_cvxlab`.
Install `cvxlab` in the same environment and extract the supplied `cvxlab_model.zip`
into this working folder. Its `sets.xlsx` already classifies the 18 products,
29 activities, 9 final-demand categories and 13 extensions. The twelve generation
activities and Electricity product have the `power` type. Other members use
`other`. The medium reporting aggregation retains the prepared reference labels. The high
reporting aggregation uses the five groups in `Aggregation II` from the supplied
workbook for products and production activities. Final-demand labels remain available
under their original categories.
The mapping below follows the current CVXLab tutorial's table names and combines
its account blocks. Use the exported sets and data together in the receiving model.
The supplied input workbook also contains the three scenarios
Reference, High_RES and High_RES_domestic and their coefficient/final-demand
shocks. `total_activity_0` is intentionally blank in the published workbook.
`to_cvxlab` updates all six reference tables, including local activity outputs,
and preserves the supplied scenario shocks.
The combined CVXLab extensions table includes the required zero block for
final-demand factors. MARIO exports no separate `VY` CSV for this exercise.

The original CVXLab configuration uses `minv(diag(X_0))` and `minv(diag(Q_0))`.
The Italian reference includes zero-output nuclear and tide/wave activities,
making `diag(X_0)` singular. The data export is complete. The receiving model needs zero-safe
coefficient normalization before the solution calls below can run.

The CVXLab calls follow its [development revision](https://github.com/cvxlab/cvxlab/tree/466b50c2c5ad8b4b2a7f02f71da5f473decbc81d). Install that revision for the optional path.

```sh
python -m pip install "cvxlab @ git+https://github.com/cvxlab/cvxlab.git@466b50c2c5ad8b4b2a7f02f71da5f473decbc81d"
```

In [ ]:
continue_in_cvxlab = False

if continue_in_cvxlab:
    model_dir = Path("cvxlab_model")
    assert (model_dir / "sets.xlsx").is_file(), "Extract cvxlab_model.zip first."
    report = db.to_cvxlab(
        model_settings=model_dir / "model_settings.xlsx",
        path=model_dir,
        mapping={
            "sets": {
                "Commodity_from": "products",
                "Commodity_to": "products",
                "Activity_from": "accounts",
                "Activity_to": "accounts",
                "Consumption category_to": "accounts",
                "Factor of production_from": "extensions",
                "Satellite account_from": "extensions",
            },
            "matrices": {
                "S": "make_0",
                "U": "use_0", "Yc": "use_0",
                "IMP_a": "imports_0", "IMP_Y": "imports_0",
                "Va": "extensions_0", "VY": "extensions_0",
                "Ea": "extensions_0", "EY": "extensions_0",
                "Xa": "total_activity_0", "Xc": "total_demand_0",
            },
        },
    )
    for name in ["pending_sets", "missing_attributes", "pending_matrices",
                 "unmatched_coordinates", "missing_values", "extra_input_rows"]:
        assert not report[name], (name, report[name])
    display(report)

### Build the receiving CVXLab model

The export prepares `model_settings.xlsx`, `sets.xlsx` and
`input_data/input_data.xlsx` in `cvxlab_model`. Follow the
[CVXLab model-building steps](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html#model-class-instance-generation)
to create the model instance.

```python
import cvxlab

model = cvxlab.Model(
    model_dir_name="cvxlab_model",
    model_settings_from="xlsx",
    use_existing_data=False,
    multiple_input_files=False,
    input_data_files_type="xlsx",
)
```

Retain the exported workbooks when CVXLab asks whether to replace existing
files. Categories, filters and scenario shocks are supplied. Resolve the zero-output
normalization described above before initializing and solving the numerical
problems. The following calls show the receiving-model workflow.

```python
model.initialize_model_environment()
model.refresh_database_and_initialize_problem(force_overwrite=True)
model.run_model(
    solution_mode="sequential",
    sequential_solution_chain=["data_processing", "impact_analysis"],
)
model.load_results_to_database(force_overwrite=True)
```

The two implementations share the input system and scenario assumptions.
The following cells complete the exercise in MARIO.

## Scenario analysis in MARIO

### High_RES

The CVXLab exercise prescribes generation shares and keeps reference input,
factor and emission coefficients fixed. MARIO's `update_supply_mix` applies
the same type of market-share update to the electricity commodity.

The source has zero reference production for nuclear and tide/wave generation
in Italy. This comparison retains their stored reference recipes, matching
the fixed-coefficient equations. Positive production with an empty recipe
contributes no inputs or direct emissions. Assign suitable technology recipes
before interpreting those activities as an energy-transition scenario.
The target shares in this teaching example are illustrative.

In [ ]:
generation = [
    name for name in db.get_index("Activity")
    if name.startswith("Production of electricity by")
    or name == "Production of electricity nec"
]
high_res_shares = dict.fromkeys(generation, 0.0)
targets = {
    "nuclear": 0.15,
    "hydro": 0.09,
    "wind": 0.38,
    "biomass and waste": 0.05,
    "solar photovoltaic": 0.31,
    "tide, wave, ocean": 0.005,
    "Geothermal": 0.015,
}
for technology, share in targets.items():
    activity, = [name for name in generation if technology in name]
    high_res_shares[activity] = share

db.update_supply_mix(
    {"IT": high_res_shares},
    scenario="High_RES",
    clone_from="baseline",
    level="Activity",
    commodities="Electricity",
    fill_missing_recipes=False,
)

### High_RES_domestic

Clone High_RES and transfer the imported-electricity requirement to the
domestic electricity row. Intermediate transfers change `u` and `imp_a`.
Final-demand transfers change `Yc` and `IMP_Y`. Imports of other products
retain their coefficients. Reset to coefficients before editing so subsequent
quantities and flows are rebuilt from the changed system.

In [ ]:
db.clone_scenario("High_RES", "High_RES_domestic")
db.reset_to_coefficients("High_RES_domestic")
u = db.query("u", scenarios="High_RES_domestic").copy()
imported = db.query("imp_a", scenarios="High_RES_domestic").copy()
demand = db.query("Yc", scenarios="High_RES_domestic").copy()
imported_final = db.query("IMP_Y", scenarios="High_RES_domestic").copy()

electricity_row, = np.flatnonzero(u.index.get_level_values(-1) == "Electricity")
import_row, = np.flatnonzero(imported.index.get_level_values(-1) == "Electricity")
u.iloc[electricity_row, :] = (
    u.iloc[electricity_row, :].to_numpy() + imported.iloc[import_row, :].to_numpy()
)
imported.iloc[import_row, :] = 0.0
demand.iloc[electricity_row, :] = (
    demand.iloc[electricity_row, :].to_numpy() + imported_final.iloc[import_row, :].to_numpy()
)
imported_final.iloc[import_row, :] = 0.0
db.update_scenarios(
    "High_RES_domestic",
    u=u,
    imp_a=imported,
    Yc=demand,
    IMP_Y=imported_final,
)

## Compare and interpret the results

MARIO resolves the same quantity model, `Xc = u @ Xa + Yc.sum(axis=1)` and
`Xa = s @ Xc`, then reconstructs supply, use, imports, factors and emissions.
Querying a matrix triggers its calculation for the selected scenario.
Sum the intermediate and direct final-demand accounts for each indicator.
The displayed scales are M EUR and kt CO2-equivalent.

In [ ]:
scenario_names = ["baseline", "High_RES", "High_RES_domestic"]
rows = {}
for scenario in scenario_names:
    matrices_for_scenario = {
        name: db.query(name, scenarios=scenario)
        for name in ["Xa", "Xc", "U", "S", "Va", "VY", "Ea", "Ec", "EY", "IMP_a", "IMP_c", "IMP_Y"]
    }
    rows[scenario] = {
        "Factors of production (M EUR)": sum(
            matrices_for_scenario[name].to_numpy().sum() for name in ["Va", "VY"]
        ) / 1e6,
        "Imports (M EUR)": sum(
            matrices_for_scenario[name].to_numpy().sum() for name in ["IMP_a", "IMP_c", "IMP_Y"]
        ) / 1e6,
        "GHG (kt CO2-eq)": sum(
            matrices_for_scenario[name].to_numpy().sum() for name in ["Ea", "Ec", "EY"]
        ) / 1e6,
    }
comparison = pd.DataFrame.from_dict(rows, orient="index")
changes = (comparison.div(comparison.loc["baseline"]) - 1.0) * 100
assert not db.query("IMP_a", scenarios="High_RES_domestic").iloc[import_row].any()
assert not db.query("IMP_Y", scenarios="High_RES_domestic").iloc[import_row].any()
display(comparison, changes.rename_axis("Change from baseline (%)"))

## Export the matrices and matching sets

For this exercise, `IMP_c` and `VY` are identically zero in every scenario.
The explicit public export list omits them and `Xa`. It includes `S`, corrected `Ea`
and the remaining reference flows; the public package contains nine matrices. The local CVXLab export above still uses
`Xa`, which remains necessary to initialize that receiving model.

Each CSV contains one MARIO matrix in long format with labelled coordinates, `Value`
and `Unit` columns. The workbook contains the transformed sets in their current order.
`S` has activities on rows and commodities on columns; `U` uses commodities on rows
and activities on columns. `Xc` provides commodity output.

Monetary flows are in EUR and GHG flows in kg CO2-equivalent. Dividing by 1,000,000
gives the M EUR and kt CO2-equivalent presentation scales used in the comparison.

In [ ]:
matrices = [
    "S", "U", "Yc", "Va", "Ea", "EY", "Xc",
    "IMP_a", "IMP_Y",
]
# Remove these generated CSVs when rerunning an earlier tutorial version.
for omitted_matrix in ["IMP_c", "VY", "Xa"]:
    previous_csv = output_dir / f"MRSUT_2022_{omitted_matrix}_2022_SUT_industry-based.csv"
    previous_csv.unlink(missing_ok=True)

db.export(
    matrices=matrices,
    scenarios=scenario_names,
    format="csv",
    split="matrix",
    path=str(output_dir),
    include_meta=False,
)
sets_path = db.export_sets(path=output_dir / "sets.xlsx")
assert sets_path.is_file()

exported = {}
for path in sorted(output_dir.glob("MRSUT_2022_*.csv")):
    frame = pd.read_csv(path)
    names = frame["Matrix"].unique().tolist()
    if len(names) == 1 and names[0] in matrices:
        assert np.isfinite(frame["Value"]).all()
        exported[names[0]] = {"File": path.name, "Rows": len(frame), "Unit": ", ".join(frame["Unit"].unique())}
assert set(exported) == set(matrices), exported
pd.DataFrame.from_dict(exported, orient="index")
comparison.to_csv(output_dir / "scenario_summary.csv", index_label="Scenario")

## Companion CVXLab tutorial

The MARIO path now contains the three scenarios, their accounting results,
nine CSV matrices with a `Scenario` coordinate, `sets.xlsx` and a summary.
The same files are published under `results` beside this notebook and in `results.zip`.
The [CVXLab tutorial](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html)
starts from the shared prepared reference and demonstrates its own model
construction and scenario calculations.

See MARIO's [documentation](https://www.marioproject.com/en/latest/),
[supply-mix guide](https://www.marioproject.com/en/latest/user_guide/transformations/supply_mix.html)
and [CVXLab export guide](https://www.marioproject.com/en/latest/user_guide/inspection/to_cvxlab.html)
for the APIs used here.

The shared baseline uses the 2022 electricity-mix update and direct electricity
intensities taken at runtime from the public EXIOBASE 3.11.2 IOT. The
published files include corrected `Ea` and omit `Xa`; the CVXLab input
package leaves `total_activity_0` blank until its local export is run.